# RAG evaluation

**Track:** Enterprise Knowledge Assistant · **Stage:** Evaluation

RAG evaluation separates three layers: retrieval quality, answer quality, and operational behavior. If you only read the final answer, you will misdiagnose failures. This notebook evaluates a small golden set with relevant document IDs, then gives a diagnostic next step for each failure pattern.

## What you will build

- A deterministic implementation that runs without API keys.
- A visible trace of evidence, decisions, and failure modes.
- A production design note explaining how this maps to real RAG libraries and systems.

## Concept map

```mermaid
flowchart TD
  C["Golden case"] --> R["Run retriever"]
  R --> M["Recall@K / Precision@K / MRR"]
  R --> A["Generate answer"]
  A --> F["Faithfulness / answer relevance"]
  M --> D["Diagnostic decision"]
  F --> D
```

## Setup

Run this notebook from the repository root, or open it in GitHub and copy cells into a local Jupyter session. The helper code lives in `src/enterprise_rag` so the notebook remains readable while the implementation stays testable.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

def show(obj):
    print(json.dumps(obj, indent=2))

The included dataset is small on purpose. In production you would expand it with supported, unsupported, stale, adversarial, permission-boundary, exact-code, paraphrase, and multi-hop questions.

In [ ]:
from src.enterprise_rag.lab_experiments import build_enterprise_chunks, evaluate_questions, diagnostic_next_step
cases = json.loads((ROOT / "datasets/enterprise_questions.json").read_text())
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
report = evaluate_questions(cases, chunks)
for row in report:
    row["diagnostic_next_step"] = diagnostic_next_step(row)
show(report)

### Metric interpretation

Recall@K asks whether labeled evidence appeared in the top K. Precision@K asks how much of the top K was useful. MRR asks how early the first relevant item appeared. Faithfulness asks whether the answer stayed inside the evidence. Cost per successful task asks whether quality improvements are worth their latency and spend.

## Deliberate failure case

Before moving on, make the system fail on purpose. Change one variable: chunk size, query wording, top-k, reranking terms, route choice, or evaluation labels. Write down whether the failure belongs to ingestion, retrieval, evidence selection, generation, authorization, or operations.

In [ ]:
# Try your own failure experiment here.
# Example: lower top_k to 1, ask an unsupported question, or remove an important query term.
from src.enterprise_rag.lab_experiments import build_enterprise_chunks
question = "What policy covers parental leave?"
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
print("Question:", question)
print("Now change the query, top_k, or chunking strategy and rerun a comparison helper.")

## Reflection questions

1. What did the simplest baseline get right?
2. What failure was invisible until you inspected the trace?
3. Which component would you improve first in production, and how would you prove it helped?
4. What should the system do when evidence is missing, unauthorized, stale, or contradictory?

## References and next reading

- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*: https://arxiv.org/abs/2005.11401
- Stanford IR book: https://nlp.stanford.edu/IR-book/
- LangChain retrieval concepts: https://docs.langchain.com/oss/python/langchain/retrieval
- LlamaIndex RAG guide: https://docs.llamaindex.ai/en/stable/understanding/rag/
- Haystack pipeline docs: https://docs.haystack.deepset.ai/docs/pipelines
- Ragas metrics: https://docs.ragas.io/en/stable/concepts/metrics/